# 🧠 LeetCode 155: Min Stack

**Pattern:** Secondary Parallel Stack

- **Created:** 2026-02-28
- **Focus:** Maintaining O(1) state calculation across mutations
- **Tags:** `stack` | `design` | `medium` | `citi-prep`

## 📖 Problem Statement

Design a stack that supports `push`, `pop`, `top`, and retrieving the minimum element in constant time.

Implement the `MinStack` class:
- `MinStack()` initializes the stack object.
- `void push(int val)` pushes the element `val` onto the stack.
- `void pop()` removes the element on the top of the stack.
- `int top()` gets the top element of the stack.
- `int getMin()` retrieves the minimum element in the stack.

You must implement a solution with `O(1)` time complexity for each function.

### Example 1:
```python
minStack = MinStack();
minStack.push(-2);
minStack.push(0);
minStack.push(-3);
minStack.getMin(); // return -3
minStack.pop();
minStack.top();    // return 0
minStack.getMin(); // return -2
```

## 🧠 Mental Model & Intuition

If you just had one standard stack, `push`, `pop`, and `top` are natively $O(1)$. Identifying the absolute minimum normally requires an $O(N)$ scan of the entire array.

**The "Buddy System" Analogy:**
Imagine every number you push onto the stack must bring a "Buddy" with it. The Buddy is a sticky note that says: *"Hi, out of everyone currently below me in the stack, plus myself, the absolute minimum value is X."*

So if you pop a number off the stack, you throw away its buddy too. The number immediately beneath it STILL has its own accurate, historically-locked sticky note showing what the minimum was at the exact moment *it* was added.

We implement this by using two parallel stacks (or storing tuples in one stack).

## 🐢 Brute Force Approach

Use a single list for the stack. Let `push`, `pop`, and `top` function as normal operations. When `getMin` is called, use Python's built-in `min(stack_list)`.

```python
class MinStackBrute:
    def __init__(self):
        self.stack = []
    def push(self, val):
        self.stack.append(val)
    def pop(self):
        self.stack.pop()
    def top(self):
        return self.stack[-1]
    def getMin(self):
        return min(self.stack) # This violates the O(1) requirement.
```
**Time:** $O(N)$ for `getMin` | **Space:** $O(N)$

In [ ]:
# Optimal Approach: Two Stacks
class MinStack:
    def __init__(self):
        self.stack = []
        self.min_stack = [] # The "Buddy"

    def push(self, val: int) -> None:
        self.stack.append(val)
        # The new minimum is the smaller of: 
        # 1. The new value being pushed.
        # 2. Whatever the minimum was just before this value arrived.
        if self.min_stack:
            val = min(val, self.min_stack[-1])
        self.min_stack.append(val)

    def pop(self) -> None:
        self.stack.pop()
        self.min_stack.pop()

    def top(self) -> int:
        return self.stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]

# Simple Test Case
m = MinStack()
m.push(-2)
m.push(0)
m.push(-3)
print("Min is:", m.getMin()) # -3
m.pop()
print("Min is now:", m.getMin()) # -2

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(1)$ for ALL operations. Evaluating `min(val, self.min_stack[-1])` is a comparison between exactly two integers, which is constant time.
- **Space Complexity:** $O(N)$ we maintain a secondary array of exactly the same depth $N$ as the primary stack array. $O(2N)$ simplifies to $O(N)$ extra space.

## 🎤 Interview Q&A

### Q1: Is there a way to optimize the space complexity further? What if we have millions of duplicates?
**Answer:** Yes! Instead of `min_stack` holding a value for *every* element, `min_stack` could only store values when a NEW, strictly smaller (or equal) element is pushed. When `pop()` happens, we check if the popped value equals the top of `min_stack`. If it is, *then* we pop `min_stack`. This reduces practical space overhead dramatically, though asymptotic worst-case is still $O(N)$.

---

### Q2: What if we stored a single variable `self.min_val` instead of a whole secondary stack?
**Answer:** That works fine for `push()` and `getMin()`, but breaks entirely on `pop()`. If our global minimum was `-5`, and we pop it off the stack, how do we know what the *previous* minimum was? We'd have to rescan the whole stack to find the new minimum, violating the $O(1)$ constraint. Overcoming `pop()` destruction is the exact reason the secondary stack is required.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Parallel Arrays/Stacks** | Maintaining two data structures in perfect sync, meaning index X in both arrays correspond to the same logical event. | `stack` and `min_stack` |
| **Constant Time** | Operation speed does not increase regardless of how large the underlying data grows. | $O(1)$ |
| **State Preservation** | Encoding historical facts at specific points in time so they can be instantly restored on regressions. | Appending the min to `min_stack` |

## 💼 The Citi Narrative

**Context:** Undo/Redo operations with risk exposure limits.

**Scenario:** A trader creates a sequence of nested portfolio "What-If" scenarios on their terminal, stacking hypothetical asset buys/sells. The UI needs to display the absolute lowest liquidity drop across the entire active sequence instantly, while allowing the trader to hit "Undo" (popping the stack back) rapidly without UI lag.

**Impact:** Implementing the MinStack pattern for the scenario manager meant that hitting "Undo" instantly rolled the system state and the minimum liquidity calculation back to perfectly accurate historical levels in constant $O(1)$ time, avoiding recalculation lag even when traders had stacked 500 deep hypothetical branches.

In [ ]:
# EXERCISE: Trace the values
# Push 5 -> Stack=[5], MinStack=[5]
# Push 7 -> Stack=[5, 7], MinStack=[5, 5] (5 is still smaller than 7)
# Push 2 -> Stack=[5, 7, 2], MinStack=[5, 5, 2] (2 is the new smallest)
print("The secondary stack effectively acts as a time-travel machine for the minimum.")

## 🎯 Summary: Key Takeaways

### The Pattern
**Secondary Parallel Stack** — Maintaining O(1) state calculation across mutations

### When to Use It
- ✅ Preserving historical bounds during time-travel or LIFO regressions.
- ✅ Constant time retrieval of global metrics without re-scanning arrays.
- ❌ Don't use when: State regressions don't occur (Use a single `min_val` Tracker instead).

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N)$ for getMin | $O(N)$ |
| Optimal | $O(1)$ for ALL | $O(N)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Secondary Parallel Stack** and you've mastered one of the most common patterns in data engineering interviews. 🚀